## Brain Tumor Detection

### [Model Training using Tensorflow]


#### Reading the dataset

In [ ]:
import os

train_dir = "../data/brain-tumor-mri-dataset/Training"
test_dir = "../data/brain-tumor-mri-dataset/Testing"

# To load and shuffle the training data
train_paths = []
train_labels = []

for label in os.listdir(train_dir):
    label_dir = os.path.join(train_dir, label)
    for img_name in os.listdir(label_dir):
        train_paths.append(os.path.join(label_dir, img_name))
        train_labels.append(label)

print(f"Number of training samples: {len(train_paths)}")
print(f"Unique labels: {set(train_labels)}")

#### Shuffling the training data

In [ ]:
from sklearn.utils import shuffle

for label in os.listdir(test_dir):
    label_dir = os.path.join(test_dir, label)
    print(f"Total test samples for label '{label}': {len(os.listdir(label_dir))}")
    for img_name in os.listdir(label_dir):
        print(f"Test image: {os.path.join(label_dir, img_name)} with label: {label}")
        train_paths.append(os.path.join(label_dir, img_name))
        train_labels.append(label)


train_paths, train_labels = shuffle(train_paths, train_labels, random_state=42)

#### Image Preprocessing (Helper Functions)


In [ ]:
import random
import numpy as np
from PIL import Image, ImageEnhance
from tensorflow.keras.preprocessing.image import load_img

1. Image Augmentation function


In [ ]:
def augment_image(image):
    image = Image.fromarray(np.uint8(image))
    image = ImageEnhance.Brightness(image).enhance(
        random.uniform(0.8, 1.2)
    )  # Random brightness
    image = ImageEnhance.Contrast(image).enhance(
        random.uniform(0.8, 1.2)
    )  # Random contrast
    image = np.array(image) / 255.0  # Normalize pixel values to [0, 1]
    return image

2. Load images and apply augmentation


In [ ]:
def open_images(paths):
    images = []
    for path in paths:
        image = load_img(path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
        image = augment_image(image)
        images.append(image)
    return np.array(images)

3. Encoding labels (convert label names to integers)


In [ ]:
def encode_label(labels):
    unique_labels = os.listdir(train_dir)  # Ensure unique labels are determined
    encoded = [unique_labels.index(label) for label in labels]
    return np.array(encoded)

4. Data generator for batching


In [ ]:
def datagen(paths, labels, batch_size=12, epochs=1):
    for _ in range(epochs):
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i : i + batch_size]
            batch_images = open_images(batch_paths)  # Open and augment images
            batch_labels = labels[i : i + batch_size]
            batch_labels = encode_label(batch_labels)  # Encode labels
            yield batch_images, batch_labels  # Yield the batch

#### Defining our training loop
[Selecting VGG16 as our Pre-trained model and using Transfer Learning to perform hyperparameter finetuning]

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16

1. Defining trainable parameters for VGG16

In [ ]:
IMAGE_SIZE = 128
num_classes = len(os.listdir(train_dir))

# Base Model
base_model = VGG16(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, weights="imagenet"
)

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze the last few layers (Conv block 5)
for layer in base_model.layers[-4:]:
    layer.trainable = True

2. Defining our final neural network

In [ ]:
from tensorflow.keras.layers import Input, Flatten, Dropout, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

In [ ]:
model = Sequential(
    [
        Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
        base_model,
        Flatten(),
        Dropout(0.3),
        Dense(128, activation="relu"),
        Dropout(0.2),   # Dropping 20% of the neurons to prevent overfitting
        Dense(num_classes, activation="softmax"),
    ]
)

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

3. Training our final neural network

In [ ]:
# Train
batch_size = 10
epochs = 15
steps = int(len(train_paths) / batch_size)

# datagen is a generator yielding (x_batch, y_batch)
train_dataset = datagen(train_paths, train_labels, batch_size=batch_size, epochs=epochs)

history = model.fit(train_dataset, epochs=epochs, steps_per_epoch=steps)

#### Plotting our results

In [ ]:
from matplotlib import pyplot as plt

1. Model Training Graph: Accuracy vs. Loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.grid(True)
plt.plot(history.history["sparse_categorical_accuracy"], ".g-", linewidth=2)
plt.plot(history.history["loss"], ".r-", linewidth=2)
plt.title("Model Training History")
plt.xlabel("epoch")
plt.xticks([x for x in range(epochs)])
plt.legend(["Accuracy", "Loss"], loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

2. Model Classification report

In [ ]:

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
from sklearn.preprocessing import label_binarize
from tensorflow.keras.models import load_model
import numpy as np

In [ ]:
# 1. Prediction on test data
test_images = open_images(test_paths)  # Load and augment test images
test_labels_encoded = encode_label(test_labels)  # Encode the test labels

# Predict using the trained model
test_predictions = model.predict(test_images)

# 2. Classification Report
print("Classification Report:")
print(classification_report(test_labels_encoded, np.argmax(test_predictions, axis=1)))

ModuleNotFoundError: No module named 'seaborn'